# Dave's Trading System: Bounce 5, Predator Play, Fibonacci & Trinity
## Tradovate Futures Edition — MES + MNQ (Micro E-mini)

**Instruments:** MES (Micro S&P 500) + MNQ (Micro Nasdaq)
**Timeframe:** 1-min bars
**Account:** DEMO4360810 (Simulated)

| Strategy | Season | Notes |
|---|---|---|
| Bounce 5 | Ranging | SHORT at 5th bounce of channel |
| Predator Play | Trending | LONG after cross below 200 MA |
| Fibonacci | All | 38.2 / 50 / 61.8% entries |
| Trinity | Confluence | A+ setup — full size |


In [9]:
# ── 1. INSTALL & IMPORTS ──────────────────────────────────────────────────────
!pip install requests pandas numpy --quiet

import requests, json, time, datetime, pytz, math, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
print('Libraries ready.')

Libraries ready.


In [10]:
# ── 2. TRADOVATE CONFIG & AUTH ────────────────────────────────────────────────
# Fill in your Tradovate demo credentials
TV_USER   = 'JaBoyWyatt'   # your Tradovate email
TV_PASS   = 'Jaboris3210!'   # your Tradovate password
TV_DEVICE = 'colab-bot-001'   # any unique string
TV_CID    = 'xyz'             # Tradovate app client ID (use 'xyz' for demo)
TV_ACCT   = 'DEMO4360810'     # your demo account name

# Demo endpoints
TV_AUTH_URL = 'https://demo.tradovateapi.com/v1/auth/accesstokenrequest'
TV_BASE_URL = 'https://demo.tradovateapi.com/v1'
TV_MD_URL   = 'https://md.tradovateapi.com/v1'   # market data

DEMO = True   # always True for sim

def tv_login():
    payload = {
        'name': TV_USER,
        'password': TV_PASS,
        'appId': 'Colab Bot',
        'appVersion': '1.0',
        'cid': TV_CID,
        'deviceId': TV_DEVICE,
        'sec': ''
    }
    r = requests.post(TV_AUTH_URL, json=payload)
    data = r.json()
    if 'accessToken' not in data:
        raise Exception(f'Login failed: {data}')
    token = data['accessToken']
    print(f'Logged in. Token: {token[:20]}...')
    return token

TV_TOKEN = tv_login()
TV_HEADERS = {'Authorization': f'Bearer {TV_TOKEN}', 'Content-Type': 'application/json'}

# Get account ID number from account name
def get_account_id():
    r = requests.get(f'{TV_BASE_URL}/account/list', headers=TV_HEADERS)
    accounts = r.json()
    for a in accounts:
        if a.get('name') == TV_ACCT:
            print(f'Account found: {a["name"]} | id={a["id"]} | balance=${a.get("netLiq","?")}')
            return a['id']
    print('Accounts available:', [a.get('name') for a in accounts])
    return accounts[0]['id']

TV_ACCT_ID = get_account_id()

Exception: Login failed: {'p-ticket': 'MzUsXzdodGtGMU9idzNHcW1DelpUUFprWjNmRDhWLW5MMkJqQ1JuNzdWUkFONjNGY0lMOTFnR0lWeVNROEtETkl6UzFsNmpzZ1M0RFhya2RhUVRYQQ', 'p-time': 15, 'p-captcha': True, 'p-message': 'Rate limit exceeded: more than 5 requests per hour'}

In [ ]:
# ── 3. FUTURES CONTRACT LOOKUP ────────────────────────────────────────────────
# Tradovate uses contract IDs. We need to find the front-month MES and MNQ.

def get_contract(symbol_root):
    """Find the active front-month contract for a symbol root like MES or MNQ."""
    r = requests.get(f'{TV_BASE_URL}/contract/suggestContracts?t={symbol_root}&l=5', headers=TV_HEADERS)
    contracts = r.json()
    if not contracts:
        raise Exception(f'No contracts found for {symbol_root}')
    # Pick first active one
    for c in contracts:
        name = c.get('name', '')
        if symbol_root in name:
            print(f'Contract: {name} | id={c["id"]} | tickSize={c.get("tickSize")} | pointValue={c.get("contractMaturityId")}')
            return c
    return contracts[0]

MES_CONTRACT = get_contract('MESM5')  # June 2025 MES
MNQ_CONTRACT = get_contract('MNQM5')  # June 2025 MNQ

# Fallback: try without month code
if not MES_CONTRACT:
    MES_CONTRACT = get_contract('MES')
if not MNQ_CONTRACT:
    MNQ_CONTRACT = get_contract('MNQ')

MES_ID = MES_CONTRACT['id']
MNQ_ID = MNQ_CONTRACT['id']
MES_NAME = MES_CONTRACT['name']
MNQ_NAME = MNQ_CONTRACT['name']

CONTRACTS = {
    'MES': {'id': MES_ID, 'name': MES_NAME, 'tick': 0.25, 'tick_value': 1.25},   # $1.25 per tick
    'MNQ': {'id': MNQ_ID, 'name': MNQ_NAME, 'tick': 0.25, 'tick_value': 0.50},   # $0.50 per tick
}
SCAN_SYMBOLS = ['MES', 'MNQ']
print(f'Ready to scan: {SCAN_SYMBOLS}')

In [ ]:
# ── 4. DATA FETCHING ──────────────────────────────────────────────────────────
import datetime as dt

def get_bars(symbol, lookback_bars=250):
    """Fetch 1-min OHLCV bars from Tradovate market data API and compute MAs."""
    contract_id = CONTRACTS[symbol]['id']

    # Tradovate uses unix timestamps in ms
    now_ms   = int(time.time() * 1000)
    start_ms = now_ms - (lookback_bars * 60 * 1000)  # lookback_bars minutes ago

    params = {
        'symbol': CONTRACTS[symbol]['name'],
        'chartDescription': json.dumps({
            'underlyingType': 'MinuteBar',
            'elementSize': 1,
            'elementSizeUnit': 'UnderlyingUnits',
            'withHistogram': False
        }),
        'timeRange': json.dumps({
            'asMuchAsElements': lookback_bars
        })
    }

    r = requests.get(f'{TV_MD_URL}/chart/getchartdata', headers=TV_HEADERS, params=params)
    if r.status_code != 200:
        print(f'  [{symbol}] Data error: {r.status_code} {r.text[:100]}')
        return None

    data = r.json()
    bars_raw = data.get('bars', data.get('s', {}).get('bars', []))

    if not bars_raw:
        print(f'  [{symbol}] No bars returned')
        return None

    rows = []
    for b in bars_raw:
        rows.append({
            'timestamp': b.get('timestamp', b.get('t', 0)),
            'open':  float(b.get('open',  b.get('o', 0))),
            'high':  float(b.get('high',  b.get('h', 0))),
            'low':   float(b.get('low',   b.get('l', 0))),
            'close': float(b.get('close', b.get('c', 0))),
            'volume':float(b.get('upVolume', b.get('v', 0))) + float(b.get('downVolume', 0)),
        })

    bars = pd.DataFrame(rows)
    bars = bars.sort_values('timestamp').reset_index(drop=True)

    # Compute MAs
    bars['ma50']  = bars['close'].rolling(50).mean()
    bars['ma200'] = bars['close'].rolling(200).mean()
    bars['slope50']  = bars['ma50'].diff(5)
    bars['slope200'] = bars['ma200'].diff(5)

    return bars

# Quick test
test = get_bars('MES', lookback_bars=50)
if test is not None:
    print(f'MES bars: {len(test)} | last close: {test.iloc[-1]["close"]}')
else:
    print('MES data fetch failed — check token or contract name')

In [ ]:
# ── 5. STRATEGY FUNCTIONS (same as Alpaca bot) ────────────────────────────────

def detect_season(bars):
    latest = bars.iloc[-1]
    slope50  = latest['slope50']  or 0
    slope200 = latest['slope200'] or 0
    sep = abs(latest['ma50'] - latest['ma200']) if pd.notna(latest['ma50']) and pd.notna(latest['ma200']) else 0
    price = latest['close']
    sep_thresh = price * 0.002
    slope_thresh = price * 0.0005
    is_ranging  = abs(slope50) < slope_thresh and sep < sep_thresh
    is_trending = abs(slope50) > slope_thresh and sep > sep_thresh
    if is_ranging:
        return {'season':1,'name':'RANGING','description':'MAs flat and intertwined. Use Bounce 5 Channel Play.','golden_cross':False,'death_cross':False}
    elif is_trending:
        direction = 'UP' if slope50 > 0 else 'DOWN'
        return {'season':2,'name':f'TRENDING {direction}','description':f'MAs separated, sloping {direction}. Use Predator Play.','golden_cross':slope50>0,'death_cross':slope50<0}
    else:
        return {'season':3,'name':'WHIPSAW','description':'Mixed signals. Stand aside or reduce size.','golden_cross':False,'death_cross':False}

def detect_bounce5(bars):
    if len(bars) < 20:
        return {'valid':False,'reason':'Not enough bars'}
    closes = bars['close'].values
    highs  = bars['high'].values
    lows   = bars['low'].values
    # Find local highs (bounces)
    bounces = []
    for i in range(2, len(closes)-2):
        if highs[i] > highs[i-1] and highs[i] > highs[i-2] and highs[i] > highs[i+1] and highs[i] > highs[i+2]:
            bounces.append((i, highs[i]))
    if len(bounces) < 5:
        return {'valid':False,'reason':f'Only {len(bounces)} bounces found, need 5'}
    b1,b2,b3,b4,b5 = [b[1] for b in bounces[-5:]]
    if not (b5 > b4 > b2):
        return {'valid':False,'reason':'Bounce 4 not higher than Bounce 2. Not ascending channel.'}
    channel_high = max(b1,b3,b5)
    channel_low  = min([lows[bounces[-5+i][0]] for i in range(5)])
    channel_range = channel_high - channel_low
    if channel_range <= 0:
        return {'valid':False,'reason':'Zero channel range'}
    angle = math.degrees(math.atan(channel_range / 20))
    if angle > 45:
        return {'valid':False,'reason':f'Channel angle {angle:.1f}° above 45°'}
    latest = bars.iloc[-1]
    entry  = round(b5, 2)
    target = round(channel_low, 2)
    stop   = round(b5 + channel_range * 0.25, 2)
    dist_t = abs(entry - target)
    dist_s = abs(stop  - entry)
    rr = round(dist_t / dist_s, 1) if dist_s > 0 else 0
    if rr < 4.0:
        return {'valid':False,'reason':f'R:R {rr}:1 below minimum 4.0:1'}
    return {'valid':True,'entry_price':entry,'target_price':target,'stop_price':stop,'risk_reward':rr,'warnings':[]}

def detect_predator_play(bars):
    if len(bars) < 20:
        return {'valid':False,'reason':'Not enough bars'}
    latest = bars.iloc[-1]
    if pd.isna(latest['ma200']):
        return {'valid':False,'reason':'MA200 not ready'}
    recent = bars.tail(20)
    crossed_below = any(
        (recent['close'].iloc[i] < recent['ma200'].iloc[i] and
         recent['close'].iloc[i-1] >= recent['ma200'].iloc[i-1])
        for i in range(1, len(recent))
    )
    if not crossed_below:
        return {'valid':False,'reason':'No recent cross below 200 MA found.'}
    cross_idx = None
    for i in range(1, len(recent)):
        if recent['close'].iloc[i] < recent['ma200'].iloc[i] and recent['close'].iloc[i-1] >= recent['ma200'].iloc[i-1]:
            cross_idx = i
    cross_bar   = recent.iloc[cross_idx]
    ma200_cross = cross_bar['ma200']
    distance_d  = abs(latest['close'] - ma200_cross)
    entry       = round(latest['close'], 2)
    target      = round(ma200_cross + distance_d * 2, 2)
    stop        = round(entry - distance_d * 0.5, 2)
    dist_t = abs(target - entry)
    dist_s = abs(entry  - stop)
    rr = round(dist_t / dist_s, 1) if dist_s > 0 else 0
    near_entry = distance_d < abs(latest['ma200'] - latest['ma50']) * 0.5
    return {'valid':True,'entry_price':entry,'target_price':target,'stop_price':stop,
            'risk_reward':rr,'ma200_at_cross':round(ma200_cross,2),
            'distance_d':round(distance_d,2),'near_entry':near_entry}

def calc_fib_levels(swing_low, swing_high):
    diff = swing_high - swing_low
    return {0.236: round(swing_high - 0.236*diff, 2),
            0.382: round(swing_high - 0.382*diff, 2),
            0.500: round(swing_high - 0.500*diff, 2),
            0.618: round(swing_high - 0.618*diff, 2),
            0.786: round(swing_high - 0.786*diff, 2)}

def detect_fib_entry(bars):
    if len(bars) < 20:
        return {'levels': {0.382:0, 0.500:0, 0.618:0}, 'current_zone': None}
    recent = bars.tail(50)
    swing_low  = recent['low'].min()
    swing_high = recent['high'].max()
    levels = calc_fib_levels(swing_low, swing_high)
    price = bars.iloc[-1]['close']
    tol = (swing_high - swing_low) * 0.01
    zone = None
    for lvl, val in levels.items():
        if abs(price - val) <= tol:
            labels = {0.382:'Lower probability', 0.500:'Moderate probability', 0.618:'HIGHEST probability — wholesale zone', 0.786:'Deep reversal zone'}
            zone = (lvl, labels.get(lvl, 'Fib level'))
            break
    return {'levels': levels, 'current_zone': zone, 'swing_low': swing_low, 'swing_high': swing_high}

def detect_candle_pattern(bars):
    if len(bars) < 3:
        return {'pattern':'Unknown','direction':'neutral'}
    c = bars.iloc[-1]; p = bars.iloc[-2]
    body       = abs(c['close'] - c['open'])
    upper_wick = c['high'] - max(c['close'], c['open'])
    lower_wick = min(c['close'], c['open']) - c['low']
    if body == 0:
        return {'pattern':'Doji','direction':'neutral'}
    if body < (c['high'] - c['low']) * 0.1:
        return {'pattern':'Doji','direction':'neutral'}
    if c['close'] > c['open'] and p['close'] > p['open'] and c['close'] > p['high'] and c['open'] > p['close']:
        return {'pattern':'Bullish Engulfing','direction':'bullish'}
    if c['close'] < c['open'] and p['close'] < p['open'] and c['close'] < p['low'] and c['open'] < p['close']:
        return {'pattern':'Bearish Engulfing','direction':'bearish'}
    if lower_wick >= 2*body and upper_wick < body:
        return {'pattern':'Hammer','direction':'bullish'}
    if upper_wick >= 2*body and lower_wick < body and c['close'] < c['open']:
        return {'pattern':'Hanging Man','direction':'bearish'}
    if c['close'] > c['open']:
        return {'pattern':'Bullish candle','direction':'bullish'}
    else:
        return {'pattern':'Bearish candle','direction':'bearish'}

def detect_trinity(bars, b5, fib, season):
    signals = []
    if b5.get('valid'):
        signals.append('Bounce5')
    if fib.get('current_zone') and fib['current_zone'][0] >= 0.618:
        signals.append('Fib_61pct')
    elif fib.get('current_zone') and fib['current_zone'][0] >= 0.500:
        signals.append('Fib_50pct')
    elif fib.get('current_zone') and fib['current_zone'][0] >= 0.382:
        signals.append('Fib_38pct')
    latest = bars.iloc[-1]
    if not pd.isna(latest.get('ma50')) and not pd.isna(latest.get('ma200')):
        sep = abs(latest['ma50'] - latest['ma200'])
        if sep < latest['close'] * 0.001:
            signals.append('MA50')
        if sep < latest['close'] * 0.002:
            signals.append('MA200')
    if len(signals) >= 3:
        return {'valid':True,'confluence':signals,'price':round(latest['close'],2),'reason':''}
    return {'valid':False,'confluence':[],'reason':f'Only {len(signals)} confluence signals ({signals})'}

print('Strategy functions loaded.')

In [ ]:
# ── 6. TRADE MANAGEMENT ───────────────────────────────────────────────────────
MAX_TRADES  = 10
DAILY_GOAL  = 200.0   # stop after $200 profit
POSITION_QTY = 1      # contracts per trade (1 MES = $5/pt, 1 MNQ = $2/pt)

class TradeManager:
    def __init__(self):
        self.daily_pnl    = 0.0
        self.trades_today = 0
        self.open_positions = {}  # symbol -> order info
        self.traded_today = set()  # prevent same symbol twice in a row
    def can_trade(self):
        return self.trades_today < MAX_TRADES and self.daily_pnl < DAILY_GOAL

mgr = TradeManager()
print(f'Trade manager ready. Max trades: {MAX_TRADES} | Daily goal: ${DAILY_GOAL}')

In [ ]:
# ── 7. ORDER PLACEMENT (with bracket stop-loss + take-profit) ────────────────

def round_to_tick(price, tick=0.25):
    """Round a price to the nearest valid tick increment."""
    return round(round(price / tick) * tick, 10)

def place_bracket_order(symbol, action, qty, stop_price, target_price):
    """
    Place a Market entry order with an OSO bracket (stop-loss + take-profit).
    action = 'Buy' (long) or 'Sell' (short)
    stop_price   = stop-loss price
    target_price = take-profit price
    """
    tick = CONTRACTS[symbol]['tick']
    stop_price   = round_to_tick(stop_price,  tick)
    target_price = round_to_tick(target_price, tick)

    # Opposite action for the exit legs
    exit_action = 'Sell' if action == 'Buy' else 'Buy'

    payload = {
        'accountSpec':  TV_ACCT,
        'accountId':    TV_ACCT_ID,
        'action':       action,
        'symbol':       CONTRACTS[symbol]['name'],
        'orderQty':     qty,
        'orderType':    'Market',
        'isAutomated':  True,
        # OSO bracket legs - fire automatically when entry fills
        'bracket1': {
            'action':    exit_action,
            'orderType': 'Stop',
            'stopPrice': stop_price,
            'orderQty':  qty,
        },
        'bracket2': {
            'action':    exit_action,
            'orderType': 'Limit',
            'price':     target_price,
            'orderQty':  qty,
        },
    }

    r = requests.post(
        f'{TV_BASE_URL}/order/placeorder',
        headers=TV_HEADERS,
        json=payload
    )
    result = r.json()

    if result.get('orderId'):
        print(f'  Bracket order placed: {action} {qty} {symbol}'
              f' | orderId={result["orderId"]}'
              f' | Stop={stop_price} | Target={target_price}')
    else:
        print(f'  Order result: {result}')
    return result


def place_long(symbol, entry, stop, target, qty=POSITION_QTY):
    """Enter LONG with stop-loss below and take-profit above."""
    tick     = CONTRACTS[symbol]['tick']
    tick_val = CONTRACTS[symbol]['tick_value']
    risk_ticks   = round((entry - stop)    / tick, 1)
    reward_ticks = round((target - entry)  / tick, 1)
    risk_dollars   = risk_ticks   * tick_val
    reward_dollars = reward_ticks * tick_val
    rr = round(reward_ticks / risk_ticks, 1) if risk_ticks > 0 else 0

    print(f'[{symbol}] Placing LONG {qty} contract(s)')
    print(f'  Entry: ~{entry} | Stop: {stop} | Target: {target}')
    print(f'  Risk: {risk_ticks} ticks (${risk_dollars:.2f}) | '
          f'Reward: {reward_ticks} ticks (${reward_dollars:.2f}) | R:R {rr}:1')

    return place_bracket_order(symbol, 'Buy', qty, stop, target)


def place_short(symbol, entry, stop, target, qty=POSITION_QTY):
    """Enter SHORT with stop-loss above and take-profit below."""
    tick     = CONTRACTS[symbol]['tick']
    tick_val = CONTRACTS[symbol]['tick_value']
    risk_ticks   = round((stop   - entry)  / tick, 1)
    reward_ticks = round((entry  - target) / tick, 1)
    risk_dollars   = risk_ticks   * tick_val
    reward_dollars = reward_ticks * tick_val
    rr = round(reward_ticks / risk_ticks, 1) if risk_ticks > 0 else 0

    print(f'[{symbol}] Placing SHORT {qty} contract(s)')
    print(f'  Entry: ~{entry} | Stop: {stop} | Target: {target}')
    print(f'  Risk: {risk_ticks} ticks (${risk_dollars:.2f}) | '
          f'Reward: {reward_ticks} ticks (${reward_dollars:.2f}) | R:R {rr}:1')

    return place_bracket_order(symbol, 'Sell', qty, stop, target)


print('Order functions ready (bracket orders with stop-loss + take-profit).')

In [ ]:
# ── 8. LIVE TRADING LOOP ──────────────────────────────────────────────────────
import time
import datetime
import pytz

LIVE_MODE     = True
SCAN_INTERVAL = 60

def is_market_open():
    ct = pytz.timezone('America/Chicago')
    now = datetime.datetime.now(ct)
    if now.weekday() >= 5:
        return False
    # Futures (ES/NQ) trade nearly 24hrs. Use RTH: 8:30 AM - 3:00 PM CT
    open_t  = now.replace(hour=8,  minute=30, second=0, microsecond=0)
    close_t = now.replace(hour=15, minute=0,  second=0, microsecond=0)
    return open_t <= now <= close_t

def refresh_token_if_needed():
    """Re-login every 18 hours to keep token fresh."""
    global TV_TOKEN, TV_HEADERS
    try:
        TV_TOKEN  = tv_login()
        TV_HEADERS = {'Authorization': f'Bearer {TV_TOKEN}', 'Content-Type': 'application/json'}
    except Exception as e:
        print(f'Token refresh failed: {e}')

print(f'Live mode: {LIVE_MODE} | Demo: {DEMO} | Scanning: {SCAN_SYMBOLS}')
print(f'Market hours: 08:30 – 15:00 CT | Max trades: {MAX_TRADES} | Goal: ${DAILY_GOAL}')

cycle       = 0
token_timer = time.time()

while True:
    ct  = pytz.timezone('America/Chicago')
    now = datetime.datetime.now(ct)

    # Refresh token every 18 hours
    if time.time() - token_timer > 64800:
        refresh_token_if_needed()
        token_timer = time.time()

    if not is_market_open():
        print(f'[{now.strftime("%I:%M %p CT")}] Market closed. Waiting...')
        for _ in range(SCAN_INTERVAL):
            time.sleep(1)
        continue

    if not mgr.can_trade():
        print(f'[{now.strftime("%I:%M %p CT")}] Limits hit — Trades={mgr.trades_today}/{MAX_TRADES} | PnL=${mgr.daily_pnl:.2f}')
        break

    cycle += 1
    print(f'\n{"="*55}\n[{now.strftime("%I:%M %p CT")}] Cycle {cycle} | Trades={mgr.trades_today}/{MAX_TRADES} | PnL=${mgr.daily_pnl:.2f}\n{"="*55}')

    for sym in SCAN_SYMBOLS:
        try:
            bars = get_bars(sym, lookback_bars=250)
            if bars is None or len(bars) < 50:
                print(f'  [{sym}] Not enough data, skipping.')
                continue

            s    = detect_season(bars)
            fib  = detect_fib_entry(bars)
            b5   = detect_bounce5(bars)
            pred = detect_predator_play(bars)
            can  = detect_candle_pattern(bars)
            tri  = detect_trinity(bars, b5, fib, s)

            latest = bars.iloc[-1]
            tick_val = CONTRACTS[sym]['tick_value']
            print(f'\n{"="*55}\n SCANNER | {sym} | 1Min | {now.strftime("%H:%M:%S")}\n{"="*55}')
            print(f'[SEASON] {s["name"]} | {s["description"]}')
            print(f'[Price] {latest["close"]:.2f} | MA50={latest["ma50"]:.2f} | MA200={latest["ma200"]:.2f}')
            print(f'  50MA: {"ABOVE" if latest["close"]>latest["ma50"] else "BELOW"} | 200MA: {"ABOVE (bull)" if latest["close"]>latest["ma200"] else "BELOW (bear)"}')
            print(f'[TICK VALUE] ${tick_val}/tick | 1 contract')

            # ── BOUNCE 5 SHORT ──
            if b5['valid']:
                ticks_profit = abs(b5['entry_price'] - b5['target_price']) / CONTRACTS[sym]['tick']
                dollar_profit = ticks_profit * tick_val
                print(f'[BOUNCE 5] SIGNAL! SHORT@{b5["entry_price"]} | Target:{b5["target_price"]} | Stop:{b5["stop_price"]} | R:R {b5["risk_reward"]}:1 | ~${dollar_profit:.0f}/contract')
                if LIVE_MODE and can['direction'] == 'bearish' and sym not in mgr.traded_today:
                    place_short(sym, b5['entry_price'], b5['stop_price'], b5['target_price'])
                    mgr.trades_today += 1
                    mgr.traded_today.add(sym)
            else:
                print(f'[BOUNCE 5] No setup — {b5["reason"]}')

            # ── PREDATOR LONG ──
            if pred['valid']:
                ticks_profit = abs(pred['target_price'] - pred['entry_price']) / CONTRACTS[sym]['tick']
                dollar_profit = ticks_profit * tick_val
                print(f'[PREDATOR] SIGNAL! LONG@{pred["entry_price"]} | Target:{pred["target_price"]} | Stop:{pred["stop_price"]} | R:R {pred["risk_reward"]}:1 | ~${dollar_profit:.0f}/contract')
                if LIVE_MODE and can['direction'] == 'bullish' and sym not in mgr.traded_today:
                    place_long(sym, pred['entry_price'], pred['stop_price'], pred['target_price'])
                    mgr.trades_today += 1
                    mgr.traded_today.add(sym)
            else:
                print(f'[PREDATOR] No setup — {pred["reason"]}')

            # ── FIBONACCI ──
            if fib['current_zone']:
                print(f'[FIBONACCI] 38.2%={fib["levels"][0.382]} | 50%={fib["levels"][0.500]} | 61.8%={fib["levels"][0.618]}')
                print(f'  >>> Price at {fib["current_zone"][0]*100:.1f}% — {fib["current_zone"][1]}')
            else:
                print(f'[FIBONACCI] 38.2%={fib["levels"][0.382]} | 50%={fib["levels"][0.500]} | 61.8%={fib["levels"][0.618]}')

            # ── TRINITY ──
            if tri.get('valid', False):
                print(f'[TRINITY] *** A+ SETUP at {tri["price"]} | Levels: {", ".join(tri["confluence"])}')
            else:
                print(f'[TRINITY] No confluence — {tri.get("reason","N/A")}')

            print(f'[CANDLE] {can["pattern"]} ({can["direction"]})')

        except Exception as e:
            import traceback
            print(f'  [{sym}] ERROR: {e}')
            traceback.print_exc()

    print(f'\n  Sleeping {SCAN_INTERVAL}s...')
    for _ in range(SCAN_INTERVAL):
        time.sleep(1)
# ─────────────────────────────────────────────────────────────────────────────

## Notes
- **MES tick value:** $1.25/tick (0.25 points)
- **MNQ tick value:** $0.50/tick (0.25 points)
- Token auto-refreshes every 18 hours
- Stop bot: Runtime → Interrupt execution
- Add symbols: update SCAN_SYMBOLS = ['MES','MNQ','...'] in a new cell
